In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [5]:
from google.colab import files
uploaded=files.upload()
type(uploaded)
print(uploaded.keys())
filename = list(uploaded.keys())[0]
print(filename)
df=pd.read_csv(filename)

Saving Healthcare.csv to Healthcare.csv
dict_keys(['Healthcare.csv'])
Healthcare.csv


In [6]:
df = pd.read_csv("Healthcare.csv")

In [7]:
df.head(5)

,Patient_ID,Age,Gender,Symptoms,Symptom_Count,Disease
0,1,29,Male,"fever, back pain, shortness of breath",3,Allergy
1,2,76,Female,"insomnia, back pain, weight loss",3,Thyroid Disorder
2,3,78,Male,"sore throat, vomiting, diarrhea",3,Influenza
3,4,58,Other,"blurred vision, depression, weight loss, muscl...",4,Stroke
4,5,55,Female,"swelling, appetite loss, nausea",3,Heart Disease


In [8]:
# Create TB target column
df["TB"] = df["Disease"].apply(lambda x: 1 if x == "Tuberculosis" else 0)


In [9]:
# Features and target
X = df["Symptoms"]
y = df["TB"]


In [10]:
# Convert symptoms text into numbers
vectorizer = CountVectorizer()
X_vectorized = vectorizer.fit_transform(X)

In [11]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized, y, test_size=0.2, random_state=42
)

In [12]:
# Train model
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [13]:
# Predictions
y_pred = model.predict(X_test)

In [14]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred)

In [15]:
print("\nMODEL ACCURACY:")
print(accuracy)


MODEL ACCURACY:
0.9666


In [16]:
import numpy as np
import joblib, os
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

# 1 = Tuberculosis, 0 = any other disease
df["TB"] = (df["Disease"] == "Tuberculosis").astype(int)
print("Total patients:", len(df))
print("TB patients:", df["TB"].sum(), f"({df['TB'].mean():.1%})")

Total patients: 25000
TB patients: 814 (3.3%)


In [17]:
# List of every unique symptom, each kept as one whole symptom
symptom_list = sorted({s.strip().lower() for row in df["Symptoms"] for s in row.split(",")})

def encode_symptoms(text):
    present = {s.strip().lower() for s in text.split(",")}
    return [1 if sym in present else 0 for sym in symptom_list]

X_symptoms = np.array([encode_symptoms(t) for t in df["Symptoms"]])

scaler = StandardScaler()
X_age = scaler.fit_transform(df[["Age"]])

gender_columns = list(pd.get_dummies(df["Gender"]).columns)
X_gender = pd.get_dummies(df["Gender"]).astype(int).values

X = np.hstack([X_symptoms, X_age, X_gender])
y = df["TB"].values
print("Number of symptoms:", len(symptom_list))
print("Final table shape:", X.shape)

Number of symptoms: 28
Final table shape: (25000, 32)


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(class_weight="balanced", n_estimators=200,
                                            max_depth=8, random_state=42, n_jobs=-1),
}

results, fitted = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    results.append({"Model": name,
                    "Accuracy": accuracy_score(y_test, pred),
                    "Precision": precision_score(y_test, pred, zero_division=0),
                    "Recall": recall_score(y_test, pred),
                    "F1": f1_score(y_test, pred, zero_division=0),
                    "ROC-AUC": roc_auc_score(y_test, prob)})
    fitted[name] = model

table = pd.DataFrame(results).set_index("Model").round(3)
table

,Accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,
Logistic Regression,0.536,0.035,0.491,0.064,0.495
Decision Tree,0.554,0.036,0.485,0.066,0.521
Random Forest,0.907,0.019,0.037,0.025,0.524


In [19]:
best_name = table["Recall"].idxmax()
best_model = fitted[best_name]
tn, fp, fn, tp = confusion_matrix(y_test, best_model.predict(X_test)).ravel()
print("Best model:", best_name)
print("TB caught correctly:", tp)
print("TB missed:", fn)
print("False alarms:", fp)
print("Healthy correct:", tn)

Best model: Logistic Regression
TB caught correctly: 80
TB missed: 83
False alarms: 2238
Healthy correct: 2599


In [20]:
os.makedirs("models", exist_ok=True)
joblib.dump(best_model, "models/tb_model.pkl")
joblib.dump(symptom_list, "models/symptom_list.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(gender_columns, "models/gender_columns.pkl")
table.to_csv("models/model_comparison.csv")
print(os.listdir("models"))

['tb_model.pkl', 'scaler.pkl', 'symptom_list.pkl', 'gender_columns.pkl', 'model_comparison.csv']


In [21]:
chosen = ["fever", "shortness of breath", "weight loss"]   # change these
age = 45
gender = "Male"

row = [1 if s in chosen else 0 for s in symptom_list]
age_scaled = scaler.transform(pd.DataFrame({"Age": [age]}))[0].tolist()
gender_row = [1 if g == gender else 0 for g in gender_columns]

x = np.array([row + age_scaled + gender_row])
print(f"TB risk: {best_model.predict_proba(x)[0][1]:.1%}")

TB risk: 57.4%


In [22]:
!zip -qr models.zip models
from google.colab import files
files.download("models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
!pip install -q streamlit
!npm install -q localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 28.2 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [25]:
from google.colab import files
files.upload()

Saving app.py to app.py


{'app.py': b'import streamlit as st\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.feature_extraction.text import CountVectorizer\nfrom sklearn.linear_model import LogisticRegression\n\n# Load dataset\n\ndf = pd.read_csv("Dataset/Healthcare.csv")\n\n# Create TB column\ndf["TB"] = df["Disease"].apply(lambda x: 1 if x == "Tuberculosis" else 0)\n\n# Features and target\nX = df["Symptoms"]\ny = df["TB"]\n\n# Convert text into vectors\nvectorizer = CountVectorizer()\nX_vectorized = vectorizer.fit_transform(X)\n\n# Train model\nX_train, X_test, y_train, y_test = train_test_split(\n    X_vectorized, y, test_size=0.2, random_state=42\n)\n\nmodel = LogisticRegression()\nmodel.fit(X_train, y_train)\n\n# Streamlit UI\nst.title("Tuberculosis Risk Prediction System")\n\nst.write("Enter patient symptoms below:")\n\n# User input\nuser_input = st.text_input("Symptoms")\n\n# Predict button\nif st.button("Predict"):\n\n    input_vector = vectorizer.transform([us

In [26]:
!ls

 app.py		  node_modules	     'TB-Prediction-System-main (1) (1).zip'
 Healthcare.csv   package.json	     'TB-Prediction-System-main (1) (2).zip'
 models		  package-lock.json  'TB-Prediction-System-main (1).zip'
 models.zip	  sample_data	      TB-Prediction-System-main.zip


In [27]:
!ls models

gender_columns.pkl    scaler.pkl	tb_model.pkl
model_comparison.csv  symptom_list.pkl


In [28]:
import urllib
print("Your tunnel password is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

!streamlit run app.py &>/content/logs.txt &
!npx localtunnel --port 8501

Your tunnel password is: 34.122.113.213
⠙⠹your url is: https://tender-hoops-retire.loca.lt
^C


In [30]:
!pkill -f streamlit

In [31]:
%%writefile app.py
import joblib
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="TB Risk Screening", page_icon="🫁")


@st.cache_resource
def load_files():
    model = joblib.load("models/tb_model.pkl")
    symptoms = joblib.load("models/symptom_list.pkl")
    scaler = joblib.load("models/scaler.pkl")
    genders = joblib.load("models/gender_columns.pkl")
    return model, symptoms, scaler, genders


model, symptom_list, scaler, gender_columns = load_files()

st.title("🫁 Tuberculosis Risk Screening System")
tab1, tab2, tab3 = st.tabs(["Prediction", "Model Performance", "About"])

with tab1:
    st.write("Fill in the patient details and click **Predict**.")

    col1, col2 = st.columns(2)
    age = col1.number_input("Age", min_value=1, max_value=100, value=30)
    gender = col2.selectbox("Gender", gender_columns)

    chosen = st.multiselect(
        "Select symptoms",
        options=[s.title() for s in symptom_list],
    )

    if st.button("Predict", type="primary"):
        if not chosen:
            st.warning("Please select at least one symptom.")
        else:
            chosen_lower = [c.lower() for c in chosen]
            row = [1 if s in chosen_lower else 0 for s in symptom_list]
            age_scaled = scaler.transform(pd.DataFrame({"Age": [age]}))[0].tolist()
            gender_row = [1 if g == gender else 0 for g in gender_columns]
            x = np.array([row + age_scaled + gender_row])

            risk = model.predict_proba(x)[0][1] * 100

            st.metric("Estimated TB risk score", f"{risk:.1f}%")
            st.progress(min(int(risk), 100))

            if risk >= 70:
                st.error("HIGH risk band - please consult a doctor.")
            elif risk >= 40:
                st.warning("MEDIUM risk band - a check-up is recommended.")
            else:
                st.success("LOW risk band.")

    st.caption("⚠️ This is a screening aid for learning purposes, "
               "not a medical diagnosis. Always consult a doctor.")

with tab2:
    st.subheader("Model comparison")
    try:
        table = pd.read_csv("models/model_comparison.csv", index_col=0)
        st.dataframe(table)
        st.bar_chart(table[["Precision", "Recall", "F1"]])
        st.write("**Recall** = out of all real TB patients, how many the "
                 "model caught. It is the most important score for screening.")
    except FileNotFoundError:
        st.info("Run the training notebook first to create the comparison table.")

with tab3:
    st.subheader("About this project")
    st.write("""
    - **Goal:** estimate TB risk from symptoms, age and gender.
    - **Models compared:** Logistic Regression, Decision Tree, Random Forest.
    - **Imbalance handling:** class weights and stratified split.
    - **Limitation:** the training dataset is synthetic, so results
      demonstrate the pipeline and are not clinically valid.
    """)

Overwriting app.py


In [32]:
!grep -c "Dataset/Healthcare.csv" app.py

0


In [33]:
import urllib
print("Your tunnel password is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

!streamlit run app.py &>/content/logs.txt &
!npx localtunnel --port 8501

Your tunnel password is: 34.122.113.213
⠙⠹your url is: https://slimy-eggs-smash.loca.lt
^C


In [34]:
!pkill -f streamlit

In [35]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.3) ...
Setting up cloudflared (2026.9.3) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [36]:
import subprocess, time, re

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501",
                  "--server.headless", "true"],
                 stdout=open("/content/logs.txt", "w"), stderr=subprocess.STDOUT)
time.sleep(5)

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in tunnel.stdout:
    print(line, end="")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("\n>>> OPEN THIS LINK:", match.group(0))
        break

2026-10-04T09:17:36Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-04T09:17:36Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-04T09:17:39Z INF +--------------------------------------------------------------------------------------------+
2026-10-04T09:17:39Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-04T09:17:39Z INF |  https://anniversary-volunteer-messenger-proceeds.tryc

In [37]:
!zip -qr final_project.zip app.py models Healthcare.csv
from google.colab import files
files.download("final_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>